In [2]:
import pandas as pd



In [17]:
print("TA-ORF")
cp = pd.read_parquet(
    "TA-ORF-BBBC037-Rohban/CellPainting/replicate_level_cp_augmented.parquet",
    engine="fastparquet"
)

print([c for c in cp.columns if "moa" in c.lower()][:20])
print(cp.columns[:30])

TA-ORF
[]
Index(['Metadata_Plate_Map_Name', 'Metadata_Plate', 'Metadata_Well',
       'Metadata_pert_id', 'Metadata_pert_type', 'Metadata_cell_id',
       'Metadata_genesymbol_mutation', 'Metadata_genesymbol',
       'Cells_AreaShape_Area', 'Cells_AreaShape_Center_X',
       'Cells_AreaShape_Center_Y', 'Cells_AreaShape_Compactness',
       'Cells_AreaShape_Eccentricity', 'Cells_AreaShape_EulerNumber',
       'Cells_AreaShape_Extent', 'Cells_AreaShape_FormFactor',
       'Cells_AreaShape_MajorAxisLength', 'Cells_AreaShape_MaxFeretDiameter',
       'Cells_AreaShape_MaximumRadius', 'Cells_AreaShape_MeanRadius',
       'Cells_AreaShape_MedianRadius', 'Cells_AreaShape_MinFeretDiameter',
       'Cells_AreaShape_MinorAxisLength', 'Cells_AreaShape_Orientation',
       'Cells_AreaShape_Perimeter', 'Cells_AreaShape_Solidity',
       'Cells_AreaShape_Zernike_0_0', 'Cells_AreaShape_Zernike_1_1',
       'Cells_AreaShape_Zernike_2_0', 'Cells_AreaShape_Zernike_2_2'],
      dtype='str')


In [24]:
print("TA-ORFD")
ge = pd.read_parquet(
    "CDRP-BBBC047-Bray/L1000/replicate_level_l1k.parquet",
    engine="fastparquet"
)

print([g for g in ge.columns if "MoA" in g.lower()][:20])
print(ge.columns[:30])

TA-ORFD
[]
Index(['Metadata_Plate', 'Metadata_pert_id', 'Metadata_pert_iname',
       'Metadata_pert_dose_micromolar', 'Metadata_cdrp_group',
       'Metadata_SMILES', '221227_x_at', '212345_s_at', '218597_s_at',
       '217140_s_at', '209253_at', '214404_x_at', '219888_at', '201225_s_at',
       '202535_at', '219499_at', '203627_at', '204087_s_at', '205227_at',
       '202388_at', '204418_x_at', '205607_s_at', '218462_at', '213417_at',
       '205963_s_at', '201804_x_at', '212851_at', '200618_at', '210151_s_at',
       '218271_s_at'],
      dtype='str')


In [26]:
import pandas as pd

paths = {
  "CDRP": "CDRP-BBBC047-Bray/CellPainting/replicate_level_cp_augmented.parquet",
  "LINCS": "LINCS-Pilot1/CellPainting/replicate_level_cp_augmented.parquet",
  "LUAD": "LUAD-BBBC041-Caicedo/CellPainting/replicate_level_cp_augmented.parquet",
  "TAORF": "TA-ORF-BBBC037-Rohban/CellPainting/replicate_level_cp_augmented.parquet",
}

def find_pert_id_col(df):
    cands = [c for c in df.columns if c.lower() in ["metadata_pert_id", "pert_id", "metadata_pert_id_dose", "metadata_pert_id_dose", "metadata_pert_id_dose"]]
    if cands: return cands[0]
    # fallback: 가장 그럴듯한 것
    for c in df.columns:
        if "pert_id" in c.lower():
            return c
    return None

ids = {}
for name, p in paths.items():
    df = pd.read_parquet(p, engine="fastparquet")
    col = find_pert_id_col(df)
    print(name, "pert_id_col:", col)
    if col is None:
        ids[name] = set()
        continue
    ids[name] = set(df[col].dropna().astype(str).unique())

common = ids["CDRP"] & ids["LINCS"] & ids["LUAD"] & ids["TAORF"]
print("4-way intersection:", len(common))

# pairwise도
for a in ids:
    for b in ids:
        if a < b:
            print(a, b, "intersection:", len(ids[a] & ids[b]))


CDRP pert_id_col: Metadata_pert_id
LINCS pert_id_col: Metadata_pert_id
LUAD pert_id_col: Metadata_pert_id
TAORF pert_id_col: Metadata_pert_id
4-way intersection: 0
CDRP LINCS intersection: 2
CDRP LUAD intersection: 0
CDRP TAORF intersection: 0
LINCS LUAD intersection: 0
LINCS TAORF intersection: 0
LUAD TAORF intersection: 0


In [27]:
common = ids["CDRP"] & ids["LINCS"]
print(common)


{'DMSO', 'BRD-K03842655-001-02-1'}


In [28]:
for name, p in {
  "LUAD":"LUAD-BBBC041-Caicedo/CellPainting/replicate_level_cp_augmented.parquet",
  "TAORF":"TA-ORF-BBBC037-Rohban/CellPainting/replicate_level_cp_augmented.parquet",
}.items():
    df = pd.read_parquet(p, engine="fastparquet")
    for c in ["Metadata_genesymbol", "Metadata_genesymbol_mutation"]:
        if c in df.columns:
            nn = df[c].notna().mean()
            nu = df[c].dropna().astype(str).nunique()
            print(name, c, "non-null ratio:", nn, "n_unique:", nu)


LUAD Metadata_genesymbol non-null ratio: 0.9479166666666666 n_unique: 135
LUAD Metadata_genesymbol_mutation non-null ratio: 1.0 n_unique: 594
TAORF Metadata_genesymbol non-null ratio: 1.0 n_unique: 194
TAORF Metadata_genesymbol_mutation non-null ratio: 1.0 n_unique: 327
